# 🚀 Fine-tune Model UMKM Deschain
**Model:** Qwen2.5-1.5B-Instruct → LoRA fine-tune → GGUF export

**Langkah:**
1. Upload `finetune/data/train.jsonl` dan `eval.jsonl` ke Colab
2. Jalankan semua cell secara berurutan
3. Download `deschain-umkm-Q4_K_M.gguf` di akhir
4. Taruh di folder `finetune/model/` di laptop
5. Jalankan dengan Ollama

**GPU:** T4 (gratis) — estimasi waktu ~15-30 menit

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install bitsandbytes
print('✅ Dependencies installed')

In [ ]:
# ── 2. Load model dengan Unsloth (4-bit quantized) ───────────────
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,        # auto-detect
    load_in_4bit=True, # hemat VRAM
)
print(f'✅ Model loaded: {MODEL_NAME}')

In [ ]:
# ── 3. Pasang LoRA adapter ───────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # rank LoRA
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f'✅ LoRA adapter attached')
print(f'   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── 4. Upload training data ──────────────────────────────────────
# Upload train.jsonl dan eval.jsonl dari laptop kamu
from google.colab import files
print('Upload train.jsonl dan eval.jsonl dari folder finetune/data/')
uploaded = files.upload()
print('✅ Files uploaded:', list(uploaded.keys()))

In [ ]:
# ── 5. Load & format dataset ─────────────────────────────────────
from datasets import load_dataset

train_dataset = load_dataset('json', data_files='train.jsonl', split='train')
eval_dataset  = load_dataset('json', data_files='eval.jsonl',  split='train')

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_dataset = train_dataset.map(format_chat)
eval_dataset  = eval_dataset.map(format_chat)

print(f'✅ Train: {len(train_dataset)} examples')
print(f'   Eval:  {len(eval_dataset)} examples')
print(f'\nContoh:')
print(train_dataset[0]['text'][:300])

In [ ]:
# ── 6. Training ──────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_strategy='no',
        output_dir='./output',
        optim='adamw_8bit',
        seed=42,
        report_to='none',
    ),
)

print('🚀 Mulai training...')
trainer_stats = trainer.train()
print(f'\n✅ Training selesai!')
print(f'   Loss akhir: {trainer_stats.training_loss:.4f}')

In [ ]:
# ── 7. Test model ────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

def chat(pertanyaan):
    messages = [
        {"role": "system", "content": "Kamu adalah asisten konsultasi Deschain untuk UMKM Indonesia."},
        {"role": "user",   "content": pertanyaan},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=512, temperature=0.7,
        do_sample=True, pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

print('Test 1:', chat('Berapa bunga KUR 2026?'))
print()
print('Test 2:', chat('Saya baru mau usaha, modal dari mana?'))
print()
print('Test 3:', chat('Apa itu Deschain?'))

In [ ]:
# ── 8. Export ke GGUF (format untuk Ollama) ──────────────────────
print('Mengekspor ke GGUF Q4_K_M (paling efisien untuk CPU)...')

model.save_pretrained_gguf(
    'deschain-umkm',
    tokenizer,
    quantization_method='q4_k_m',  # ~1GB, bagus untuk CPU
)

import os
gguf_files = [f for f in os.listdir('.') if f.endswith('.gguf') or
              any(f.endswith(ext) for ext in ['.gguf'] ) ]
gguf_path = 'deschain-umkm/deschain-umkm-unsloth.Q4_K_M.gguf'

if os.path.exists(gguf_path):
    size_mb = os.path.getsize(gguf_path) / 1024 / 1024
    print(f'✅ GGUF tersimpan: {gguf_path} ({size_mb:.0f} MB)')
else:
    # cari file gguf
    for root, dirs, files in os.walk('.'):
        for f in files:
            if f.endswith('.gguf'):
                gguf_path = os.path.join(root, f)
                print(f'✅ GGUF: {gguf_path}')
                break

In [ ]:
# ── 9. Download file GGUF ke laptop ──────────────────────────────
from google.colab import files
print(f'Download {gguf_path}...')
print('Taruh file hasil download di: finetune/model/')
files.download(gguf_path)